# EXERCISE SOLUTION

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

In [1]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [5]:
# Constants

MODEL = "llama3.2"

In [6]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [7]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Connec

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [8]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [9]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

## Messages

The API from Ollama expects the same message format as OpenAI:

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

In [10]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Time to bring it together - now with Ollama instead of OpenAI

In [11]:
# And now: call the Ollama function instead of OpenAI

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [12]:
summarize("https://edwarddonner.com")

'**Summary**\n\n* Edward Donner is a co-founder and CTO of Nebula.io, applying AI to help people discover their potential.\n* He previously founded and CEO-ed an AI startup untapt, which was acquired in 2021.\n\n**News/Announcements**\n\n* The Complete Agentic AI Engineering Course will be released on April 21, 2025.\n* A LLM Workshop – Hands-on with Agents – is available as resources (date: December 21, 2024).\n* Welcome message for SuperDataScientists! (November 13, 2024).'

In [13]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [14]:
display_summary("https://edwarddonner.com")

# Website Summary
### About the Owner
Edward Donner is a writer, DJ, and tech enthusiast who co-founded Nebula.io, an AI startup that aims to help people discover their potential. He previously founded another AI startup, untapt, which was acquired in 2021.

### News and Announcements
#### Upcoming Events:
* **April 21, 2025**: The Complete Agentic AI Engineering Course
* **January 23, 2025**: LLM Workshop – Hands-on with Agents – resources
* **December 21, 2024**: Welcome, SuperDataScientists!
* **November 13, 2024**: Mastering AI and LLM Engineering – Resources

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [15]:
display_summary("https://cnn.com")

**Summary of the Website**
==========================

The website is a news portal provided by CNN, featuring breaking news, videos, and various categories such as politics, business, entertainment, sports, and more.

### Featured Sections

*   **Space and Science**: Articles about recent discoveries in space exploration, climate change, and scientific breakthroughs.
*   **Global Travel**: Travel guides, news, and features related to destinations around the world.
*   **Global Business**: Updates on market trends, economic news, and business strategies.

### Recent News

*   "Trump's Middle East trip: Analysis of the president's speech and its implications."
*   "Prices are falling. But it might be for bad reasons."
*   "Celine Dion makes surprise video appearance at Eurovision."

### Notable Articles

*   "The childhood nickname that still haunts Beyoncé's mom and shaped how she parents"
*   "Did 8 Italian physicists make the perfect cacio e pepe? I tested their experiment to find out"

### Live Updates

*   "US President Announces Deals with Qatar after Greeted with Mounted Camels and Red Cybertrucks"
*   "UN Aid Chief Calls on Israel to Allow Humanitarian Aid into Gaza"

Overall, this website provides a comprehensive platform for staying informed about current events from around the world.

In [16]:
display_summary("https://anthropic.com")

**Summary of Anthropic Website**
=====================================

### Mission and Philosophy

* Build AI to serve humanity's long-term well-being.
* Design powerful technologies with human benefit at their foundation.

### Products and Solutions

* Claude: an AI model that enables human-AI collaboration.
* API Platform for building custom experiences using Claude.
* Various pricing plans, including enterprise and education options.

### Research and Transparency

* Anthropic Economic Index: a metric to track societal impacts of AI.
* Core Views on AI Safety: guidelines for responsible AI development.
* Responsible scaling policy: measures to ensure safe AI deployment.

### News and Announcements

* Claude 3.7 Sonnet is now available, the most intelligent AI model yet.
* Anthropic's ISO 42001 certification: a milestone in safety standards.
* Recent research articles on alignment science, Model Context Protocol, and more.

### Community Engagement

* Join the conversation with Claude, a conversational AI assistant.
* Learn to build with Claude through Anthropic Academy.
* Explore open roles and collaborate with experts in safe AI development.

### About Us

* Meet the team behind Anthropic's mission-driven approach.
* Learn about the company's values, mission, and history.
* Discover opportunities to partner with Anthropic.

# Sharing your code

I'd love it if you share your code afterwards so I can share it with others! You'll notice that some students have already made changes (including a Selenium implementation) which you will find in the community-contributions folder. If you'd like add your changes to that folder, submit a Pull Request with your new versions in that folder and I'll merge your changes.

If you're not an expert with git (and I am not!) then GPT has given some nice instructions on how to submit a Pull Request. It's a bit of an involved process, but once you've done it once it's pretty clear. As a pro-tip: it's best if you clear the outputs of your Jupyter notebooks (Edit >> Clean outputs of all cells, and then Save) for clean notebooks.

PR instructions courtesy of an AI friend: https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c